# Drug Binding Energy with VQE (Molecular Optimization)

In this module, you will:
- Understand the application of the **Variational Quantum Eigensolver (VQE)** in quantum chemistry — the leading near-term algorithm for computing molecular ground state energies.
- Learn how to define molecular structures and calculate electronic integrals using the `MolecularDriver`.
- Compile the **Unitary Coupled Cluster Singles and Doubles (UCCSD)** ansatz — a chemistry-inspired variational circuit.
- Solve for the ground state energy of a molecular system using the QpiAI Quantum SDK.
- Analyze the optimization convergence and interpret the computed energy.

## VQE in Quantum Chemistry

Calculating molecular ground state energies is a critical step in **drug discovery** and **material design**. Classical methods scale exponentially with system size, making quantum computers a natural fit.

VQE solves the electronic Schrödinger equation by minimizing the expectation value of the electronic Hamiltonian:

$$E(\theta) = \frac{\langle \Psi(\theta) | H | \Psi(\theta) \rangle}{\langle \Psi(\theta) | \Psi(\theta) \rangle}$$

## VQE vs QAOA

| Feature | VQE | QAOA |
|:---|:---|:---|
| Circuit structure | Generic parameterized ansatz (e.g. UCCSD) | Alternating cost + mixer layers |
| Target use case | General ground-state energy / molecular simulation | Combinatorial / discrete optimization |
| Key parameters | Ansatz-dependent (excitation amplitudes) | $(\gamma, \beta)$ per layer |

VQE with a **chemistry-inspired ansatz** like UCCSD is the natural choice for molecular energy calculations.

**Key Insight:** Each qubit represents one spin-orbital of the molecule. The UCCSD ansatz prepares a trial state by applying single and double electron excitations to the Hartree-Fock reference state, and a classical optimizer tunes the excitation amplitudes to minimize the energy.

## 1. Setup & Authentication

We reuse the same authentication pattern as previous modules: load `API_KEY` from `.env` and authenticate with `QpiAIQuantumAuth`.

In [1]:
import os

from dotenv import load_dotenv
from qpiai_quantum import QpiAIQuantumAuth

load_dotenv("./qcloud.env") # This path should point to the env file containing your API key.

QpiAIQuantumAuth.login(os.getenv("API_KEY"))
user_info = QpiAIQuantumAuth.me()

print(f"✅ Authenticated successfully as: {user_info.get('name', 'User')} ({user_info.get('email', '')})")

✅ Authenticated successfully as: Test Advanced User (test_advanced@qpiai.tech)


## 2. SDK Primitives Used (This Module)

- `MolecularDriver` — runs PySCF under the hood (or uses a pre-computed fallback database) to perform Hartree-Fock calculations and return electronic integrals.
- `MolecularProperties` — data structure containing `n_electrons`, `n_orbitals`, `nuclear_repulsion_energy`, `hf_energy`, `one_body_integrals`, and `two_body_integrals`.
- `jordan_wigner` — maps fermionic Hamiltonians to qubit Pauli operators using the Jordan-Wigner transformation.
- `uccsd_ansatz` — compiles a Unitary Coupled Cluster Singles and Doubles variational circuit template.
- `VQESolver` — minimizes the molecular expectation value using classical optimizers.
- **Solver configuration:** `n_qubits`, `ansatz` (callable or string), `optimizer`, `max_iterations`, `initial_point`.
- **Execution:** `solver.run(hamiltonian, method, device_name)`.
- **Results:** `result.optimal_energy`, `result.energy_history`, `result.optimal_parameters`.

We will first define the molecule, then map its Hamiltonian, compile the UCCSD ansatz, and run VQE optimization.

## 3. Theory: Fermionic Operators to Qubit Mapping

Electronic Hamiltonians are originally represented in terms of **fermionic creation** ($a_i^\dagger$) and **annihilation** ($a_j$) operators. These obey anticommutation relations:

$$\{a_i, a_j^\dagger\} = \delta_{ij}, \quad \{a_i, a_j\} = 0$$

Qubits obey different commutation relations, so we must map the fermionic operators to **Pauli strings** using a transformation.

### Jordan-Wigner Transformation

The Jordan-Wigner mapping converts each fermionic operator to a string of Pauli operators:

$$a_j^\dagger \rightarrow \frac{1}{2} \left( \prod_{k=0}^{j-1} -Z_k \right) (X_j - i Y_j)$$

### The Electronic Hamiltonian

The full second-quantized electronic Hamiltonian is:

$$H = E_{\text{nuc}} + \sum_{pq} h_{pq}\, a_p^\dagger a_q + \frac{1}{2} \sum_{pqrs} h_{pqrs}\, a_p^\dagger a_q^\dagger a_s a_r$$

where:
- $E_{\text{nuc}}$ — nuclear repulsion energy (classical constant)
- $h_{pq}$ — one-body integrals (kinetic + nuclear attraction)
- $h_{pqrs}$ — two-body integrals (electron-electron repulsion)

### UCCSD Ansatz

The Unitary Coupled Cluster ansatz prepares the trial state:

$$|\Psi(\theta)\rangle = e^{T(\theta) - T^\dagger(\theta)} |\text{HF}\rangle$$

where $T(\theta) = T_1(\theta) + T_2(\theta)$ includes:
- **Singles:** $T_1 = \sum_{ia} t_{ia}\, a_a^\dagger a_i$
- **Doubles:** $T_2 = \sum_{ijab} t_{ijab}\, a_a^\dagger a_b^\dagger a_j a_i$

Each excitation amplitude $t$ is a variational parameter optimized by VQE.

## 4. Define the Molecule & Map the Hamiltonian

We define a Hydrogen molecule ($H_2$) at its equilibrium bond length ($0.735$ Å) using the **STO-3G** minimal basis set.

| Parameter | Value | Description |
|:---|:---|:---|
| Molecule | $H_2$ | Hydrogen molecule |
| Bond length | 0.735 Å | Near-equilibrium geometry |
| Basis set | STO-3G | Minimal basis (2 spatial orbitals → 4 spin-orbitals) |
| Electrons | 2 | Two valence electrons |
| Qubits | 4 | One per spin-orbital |

The workflow is:
1. `MolecularDriver.run()` → `MolecularProperties` (integrals + energies)
2. `MolecularProperties.get_fermionic_hamiltonian()` → `FermionOperator`
3. `jordan_wigner(fermion_op)` → list of Pauli string terms

In [2]:
import numpy as np
from qpiai_quantum.applications.chemistry import MolecularDriver, jordan_wigner

# Define H2 geometry and basis set
geometry = "H 0.0 0.0 0.0; H 0.0 0.0 0.735"
driver = MolecularDriver(geometry=geometry, basis="sto-3g")

# Step 1: Run Hartree-Fock to get molecular properties (integrals, energies)
properties = driver.run()

print(f"Molecule: H2 (STO-3G)")
print(f"Electrons: {properties.n_electrons}")
print(f"Spin-orbitals (qubits): {properties.n_orbitals}")
print(f"Nuclear repulsion energy: {properties.nuclear_repulsion_energy:.6f} Hartree")
print(f"Hartree-Fock energy: {properties.hf_energy:.6f} Hartree")

# Step 2: Get fermionic Hamiltonian from integrals
ferm_ham = properties.get_fermionic_hamiltonian()

# Step 3: Map to qubit Pauli operators using Jordan-Wigner
qubit_ham_terms = jordan_wigner(ferm_ham)

# Wrap for VQESolver (requires .get_hamiltonian_terms() interface)
class ChemistryHamiltonian:
    def get_hamiltonian_terms(self):
        return qubit_ham_terms

hamiltonian = ChemistryHamiltonian()
terms = hamiltonian.get_hamiltonian_terms()

print(f"\nTotal Hamiltonian terms (Pauli strings): {len(terms)}")
print(f"\nFirst 10 terms:")
for i, (ops, coeff) in enumerate(terms[:10]):
    if ops:
        op_str = ' '.join([f"{op}_{qi}" for qi, op in ops])
    else:
        op_str = "Identity (constant)"
    print(f"  Term {i:2d}: {op_str:35s} coeff={coeff:9.6f}")

Molecule: H2 (STO-3G)
Electrons: 2
Spin-orbitals (qubits): 4
Nuclear repulsion energy: 0.719969 Hartree
Hartree-Fock energy: -1.116999 Hartree

Total Hamiltonian terms (Pauli strings): 19

First 10 terms:
  Term  0: Identity (constant)                 coeff=-0.090579
  Term  1: Z_0                                 coeff= 0.172184
  Term  2: Z_1                                 coeff= 0.172184
  Term  3: Z_2                                 coeff=-0.225753
  Term  4: Z_3                                 coeff=-0.225753
  Term  5: Z_0 Z_1                             coeff= 0.168928
  Term  6: Z_0 Z_2                             coeff= 0.120913
  Term  7: Z_0 Z_3                             coeff= 0.166145
  Term  8: Y_0 X_1 Y_2 X_3                     coeff=-0.022616
  Term  9: Y_0 Y_1 X_2 X_3                     coeff=-0.022616


## 5. Compile the UCCSD Variational Ansatz

We construct the parameterized UCCSD quantum circuit. For $H_2$ with 4 spin-orbitals and 2 electrons:

- **Single excitations:** Spin-conserving transitions $|\text{occ}\rangle \rightarrow |\text{virt}\rangle$ (e.g., orbital 0→2, 1→3)
- **Double excitations:** Pair transitions $(i,j) \rightarrow (a,b)$

Each excitation is compiled into a sequence of CNOT chains and parameterized $R_z$ rotations via the Jordan-Wigner mapping.

| Parameter | Value | Description |
|:---|:---|:---|
| `n_qubits` | 4 | Number of spin-orbitals |
| `n_electrons` | 2 | Number of active electrons |
| `mapping` | `'jordan_wigner'` | Fermion-to-qubit mapping |

In [3]:
from qpiai_quantum.applications.chemistry import uccsd_ansatz
from qpiai_quantum.algorithms.opt.solvers.vqe import VQESolver

# Generate UCCSD ansatz circuit
ansatz_circuit = uccsd_ansatz(
    n_qubits=properties.n_orbitals,
    n_electrons=properties.n_electrons,
    mapping="jordan_wigner"
)

print(f"Compiled UCCSD circuit with {ansatz_circuit.num_qubits} qubits")

# Determine number of variational parameters
vqe_temp = VQESolver(n_qubits=properties.n_orbitals, ansatz=lambda n: ansatz_circuit)
n_params = vqe_temp._count_parameters()
print(f"Number of variational parameters: {n_params}")
print(f"\nNote: H2 has 2 single excitations and 1 double excitation,")
print(f"each compiled into multiple parameterized Pauli rotations.")

Compiled UCCSD circuit with 4 qubits
Number of variational parameters: 12

Note: H2 has 2 single excitations and 1 double excitation,
each compiled into multiple parameterized Pauli rotations.


## 6. Configure VQE Solver

We create a `VQESolver` with the following settings:

| Parameter | Value | Description |
|:---|:---|:---|
| `n_qubits` | 4 | Number of spin-orbitals (qubits) |
| `ansatz` | UCCSD (callable) | Chemistry-inspired variational ansatz |
| `optimizer` | `'cobyla'` | COBYLA — gradient-free constrained optimizer |
| `max_iterations` | 40 | Number of classical optimization iterations |
| `initial_point` | zeros | Start all excitation amplitudes at zero (Hartree-Fock state) |

Starting from the zero-parameter point corresponds to the Hartree-Fock reference state. The optimizer then adjusts the excitation amplitudes to lower the energy below the HF value.

In [4]:
initial_point = np.zeros(n_params)

vqe = VQESolver(
    n_qubits=properties.n_orbitals,
    ansatz=lambda n: ansatz_circuit,
    optimizer="cobyla",
    max_iterations=40,
    initial_point=initial_point,
    verbose=False,
)

print(f"\nVQE Configuration:")
print(f"  Qubits: {properties.n_orbitals}")
print(f"  Ansatz: UCCSD (custom)")
print(f"  Optimizer: {vqe.optimizer.upper()}")
print(f"  Max Iterations: {vqe.max_iterations}")
print(f"  Parameters: {n_params}")
print(f"  Initial point: all zeros (Hartree-Fock reference)")


VQE Configuration:
  Qubits: 4
  Ansatz: UCCSD (custom)
  Optimizer: COBYLA
  Max Iterations: 40
  Parameters: 12
  Initial point: all zeros (Hartree-Fock reference)


## 7. Run VQE Optimization

We now execute the VQE algorithm on QpiAI's local statevector simulator.

The algorithm will:
1. Initialize excitation amplitudes at zero (Hartree-Fock state).
2. Build the UCCSD circuit with current parameters.
3. Compute the exact statevector and evaluate $\langle H \rangle$.
4. Update parameters using the COBYLA optimizer.
5. Repeat until convergence or `max_iterations`.

> **Simulator default:** `device_name='QpiAI-QSV-Local'` uses the local statevector simulator. To run on a QPU, change to `device_name='QpiAI-Indus-1'`.

**Expected result:** The VQE ground state energy should be **lower** than the Hartree-Fock energy ($-1.117$ Hartree), approaching the exact value of $\approx -1.137$ Hartree.

In [5]:
result = vqe.run(
    hamiltonian=hamiltonian,
    method="statevector",
    device_name="QpiAI-QSV-Local"
)

print(f"\n" + "="*70)
print("VQE RESULTS SUMMARY")
print("="*70)
print(f"Optimized Ground State Energy: {result.optimal_energy:.6f} Hartree")
print(f"Hartree-Fock Reference Energy: {properties.hf_energy:.6f} Hartree")
print(f"Energy improvement over HF:    {abs(result.optimal_energy - properties.hf_energy):9.6f} Hartree")
print(f"Nuclear Repulsion Energy:       {properties.nuclear_repulsion_energy:9.6f} Hartree")
print(f"\nNote: Exact FCI ground state for H2/STO-3G is ~-1.137 Hartree.")
print(f"The VQE result is below HF, confirming electron correlation is captured.")


VQE RESULTS SUMMARY
Optimized Ground State Energy: -1.122609 Hartree
Hartree-Fock Reference Energy: -1.116999 Hartree
Energy improvement over HF:     0.005610 Hartree
Nuclear Repulsion Energy:        0.719969 Hartree

Note: Exact FCI ground state for H2/STO-3G is ~-1.137 Hartree.
The VQE result is below HF, confirming electron correlation is captured.


## 8. Interpret Results & Convergence

Let's analyze the optimization convergence and interpret the energy landscape:

- **Hartree-Fock energy** ($E_{\text{HF}}$): The mean-field approximation — each electron moves in the average field of all others. This is the starting point.
- **VQE energy** ($E_{\text{VQE}}$): Includes electron correlation effects captured by the UCCSD ansatz. Should be $E_{\text{VQE}} < E_{\text{HF}}$ by the **variational principle**.
- **Exact (FCI) energy**: For H2/STO-3G, the exact ground state energy is $\approx -1.137$ Hartree.

In [6]:
exact_fci_energy = -1.137  # Known FCI ground state for H2/STO-3G
correlation_energy = exact_fci_energy - properties.hf_energy
recovered = (result.optimal_energy - properties.hf_energy) / correlation_energy * 100

print(f"\n" + "="*70)
print("ENERGY COMPARISON")
print("="*70)
print(f"\n  Hartree-Fock energy:     {properties.hf_energy:.6f} Hartree  (mean-field, no correlation)")
print(f"  VQE (UCCSD) energy:      {result.optimal_energy:.6f} Hartree  (includes correlation)")
print(f"  Exact (FCI) energy:      ~{exact_fci_energy:.6f} Hartree  (full configuration interaction)")
print(f"\n  Correlation energy recovered: {recovered:.2f}%")

variational_check = result.optimal_energy < properties.hf_energy
print(f"\nThe VQE result satisfies the variational principle: E_VQE < E_HF {'✅' if variational_check else '❌'}")


ENERGY COMPARISON

  Hartree-Fock energy:     -1.116999 Hartree  (mean-field, no correlation)
  VQE (UCCSD) energy:      -1.122609 Hartree  (includes correlation)
  Exact (FCI) energy:      ~-1.137000 Hartree  (full configuration interaction)

  Correlation energy recovered: 28.05%

The VQE result satisfies the variational principle: E_VQE < E_HF ✅
